# Буткемп, день 25 — BTC/ETH perpetual market maker: интеграция Week 4

Собираем один объект из десяти переключателей дней 16–19 на single-venue паре Bybit ETHUSDT (котируемая нога) / BTCUSDT (хедж), прогоняем base / stressed / crisis, девять leave-one-out абляций, шесть отчётов из лога, сравнение с Day16-базой и объяснение interaction gap. Отдельно — оптимизация по тарифам: на каких комиссиях стратегия даёт Sharpe ≥ 1.

Данные: Bybit, 100-мс top-of-book и трейды, 2026-07-01 12:00–14:00 UTC. Тик ETH $0.01, BTC $0.1. Хедж — ratio 1:1 по долларовому нотионалу: $h_t=\overline{ETH/BTC}$, $h\approx0.0268$ BTC на 1 ETH; проверка: $h\cdot BTC\approx ETH$, а $1$ доллар движения BTC даёт $2.68$ ETH-тика. Котируем ETH, хеджируем BTC на той же бирже.

**Главный результат.** При выбранном тире 0.05 bp maker / 0.10 bp taker полная стратегия даёт **+180 / +1048 / −705** тиков в час в base / stressed / crisis (на всём окне +43: crisis уводит в минус). В base и stressed она лучше Day16-базы (−225 / +446), но в crisis хуже базы (−605): режимный overlay не валидируется в самом волатильном бакете. Хвост инвентаря шире базы (p95 1.42 против 0.88 ETH в stressed; 1.38 против 0.92 в base), поэтому hard-критерий «tail no wider than baseline» **не проходит**: часть прироста — leverage, а не улучшение. Interaction gap меняет знак: в base сумма leave-one-out дельт −190 тика/ч против фактических +406, в stressed +2052 против +602. В base слои complement, в stressed substitute: width, modes и ladder тянут вверх, toxicity их частично перекрывает, а crisis-взаимодействия отрицательны.

## 1. Архитектура: десять переключателей и порядок

Порядок слоёв фиксирован: health → hard limits → width floor → inventory skew → break state → alpha skew → flow term → quoting mode → toxicity filter → placement. Health и width floor — ранние выходы; risk controls выше views, slow views выше fast; placement последний. В конфиге десять переключателей; абляции — девять: восемь слоёв из слайда (flow, placement, adaptive width, ladder, toxicity, modes, hedge urgency, inventory skew) плюс alpha. Alpha добавлен в сумму, иначе база не равна full минус перечисленные восемь, и interaction gap включал бы его вклад.

Режимы рынка — **ex-post** стратификация по 5-минутной realized vol ETH: base — 8 самых спокойных бакетов, stressed — средние 8, crisis — 8 самых волатильных. Это ярлык для отчёта, а не вход стратегии: внутри прогона mode, width и toxicity используют только trailing vol предыдущего 5-минутного бакета; первый бакет пропускается, чтобы не заглянуть в собственное будущее. Flow-терм — упрощённый mid-momentum proxy по BTC (rolling Δmid), не depth-OFI; это оговорено в журнале оптимизма. Все прогоны детерминированы (seed=7 — задел под будущий RNG, сейчас случайность не используется).

Тарифы: базовый расчёт — 0.05 bp maker / 0.10 bp taker (верхний MM-тир, подобранный в §6); retail 0.1/2 bp в этой паре убыточен, и это отдельный вывод.

In [1]:
import duckdb, numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220)
D='/Users/b.kotov/Downloads/'; con=duckdb.connect()
W=600; STEP=50; HOLD=100; ETICK=0.01; BTICK=0.1; SEED=7

def dt_ns(x): return pd.DatetimeIndex(pd.to_datetime(x)).as_unit('ns').astype('int64').values
def panel(asset):
    return con.execute(f"""SELECT event_time, mid, best_bid, best_ask, best_bid_amount, best_ask_amount
      FROM read_parquet('{D}l2_executable_quotes_100ms_10000.parquet') WHERE asset='{asset}' AND exchange='Bybit'
      AND event_time>=TIMESTAMPTZ '2026-07-01 12:00:00+00' AND event_time<TIMESTAMPTZ '2026-07-01 14:00:00+00' ORDER BY event_time""").df().reset_index(drop=True)
def trades(sym):
    return con.execute(f"""SELECT event_time, aggressor_side, price, amount FROM read_parquet('{D}bybit_anon_trades_20260701_20260703.parquet')
      WHERE symbol='{sym}' AND event_time>=TIMESTAMPTZ '2026-07-01 12:00:00+00' AND event_time<TIMESTAMPTZ '2026-07-01 14:00:00+00' ORDER BY event_time""").df()
def map_tr(tr, pt):
    tt=dt_ns(tr.event_time); idx=np.searchsorted(pt,tt,'right')-1; ok=(idx>=0)&(idx<len(pt))
    return tt[ok],tr.aggressor_side.values.astype(int)[ok],tr.price.values[ok],tr.amount.values[ok],idx[ok]

e=panel('ETH'); b=panel('BTC')
m=pd.merge_asof(e[['event_time']], b[['event_time','mid','best_bid','best_ask','best_bid_amount','best_ask_amount']],
                on='event_time',direction='backward',tolerance=pd.Timedelta('1s'),suffixes=('','_b')).ffill()
n=len(e); pt=dt_ns(e.event_time)
et=e.mid.values; eb=e.best_bid.values; ea=e.best_ask.values; ebs=e.best_bid_amount.values; eas=e.best_ask_amount.values
bt=m.mid.values; bb=m.best_bid.values; ba=m.best_ask.values; bbs=m.best_bid_amount.values; bas=m.best_ask_amount.values
h=pd.Series(et/bt).rolling(W,min_periods=W).mean().shift(1).values
mu=pd.Series(et-h*bt).rolling(W,min_periods=W).mean().shift(1).values
spread=et-h*bt
sig=pd.Series(spread).rolling(W,min_periods=W).std().shift(1).values
z=(spread-mu)/np.maximum(sig,1e-9)
bticks=bt/BTICK
ofi=pd.Series(np.diff(bticks,prepend=bticks[0])).rolling(10,min_periods=1).sum().values
vol=pd.Series(np.diff(et,prepend=et[0])**2).groupby(np.arange(n)//3000).sum().pow(.5).values
nb=n//3000; order=np.argsort(vol)
REG={'base':set(order[:8].tolist()),'stressed':set(order[8:16].tolist()),'crisis':set(order[16:].tolist()),'all':set(range(nb))}
qmed=float(np.nanmedian(np.r_[ebs,eas]))
# trailing, point-in-time estimators
r=np.diff(et,prepend=et[0])
vol_bucket=pd.Series(r**2).groupby(np.arange(n)//3000).sum().pow(.5).values
vol_step=np.repeat(pd.Series(vol_bucket).shift(1).fillna(vol_bucket[0]).values,3000)[:n]
vmean=pd.Series(vol_step).expanding(min_periods=1).mean().shift(1).values
vstd=pd.Series(vol_step).expanding(min_periods=1).std().shift(1).values
volz_t=(vol_step-vmean)/np.maximum(vstd,1e-9)
qmed_b=pd.Series(ebs).rolling(W,min_periods=W).median().shift(1).values
qmed_a=pd.Series(eas).rolling(W,min_periods=W).median().shift(1).values
ofi_std=pd.Series(ofi).rolling(W,min_periods=W).std().shift(1).values
Te=trades('ETHUSDT'); Tb=trades('BTCUSDT')
tt,sd_e,px_e,am_e,idx_e=map_tr(Te,pt); px_e=px_e/ETICK
ttb,sd_b,px_b,am_b,idx_b=map_tr(Tb,pt); px_b=px_b/BTICK
print('панели %d, трейды ETH %d / BTC %d, h=%.5f BTC/ETH'%(n,len(tt),len(ttb),np.nanmedian(h)))
print('unit check: h*BTC=%.1f vs ETH=%.1f; $1 BTC move -> %.2f ETH ticks'%(
    np.nanmedian(h)*np.nanmedian(bt),np.nanmedian(et),np.nanmedian(h)/ETICK))
print('5-мин vol по бакетам:',np.round(vol,2))
print('регимы: base %d, stressed %d, crisis %d'%(len(REG['base']),len(REG['stressed']),len(REG['crisis'])))

панели 70878, трейды ETH 490747 / BTC 454717, h=0.02682 BTC/ETH
unit check: h*BTC=1571.6 vs ETH=1570.6; $1 BTC move -> 2.68 ETH ticks
5-мин vol по бакетам: [1.94 2.34 2.76 2.4  1.59 1.68 2.9  2.51 2.01 2.37 2.73 2.5  2.82 1.65
 1.65 2.18 2.64 5.71 5.18 5.58 5.14 6.46 4.92 2.57]
регимы: base 8, stressed 8, crisis 8


## 2. Пять предсказаний — до абляций

1. **Toxicity × ladder — substitution.** Фильтр срезает именно внешние филлы, которые добавляет ladder; вклад ladder при включённом фильтре будет меньше, чем без него.
2. **Adaptive width × mode switching — double application.** Оба расширяют котировки в стрессе; вместе дадут больше времени вне рынка, чем каждый по отдельности, и в stressed net может оказаться ниже, чем при одном из них.
3. **Hedge urgency × inventory skew — substitution.** Агрессивный хедж держит инвентарь маленьким, и skew становится меньше работы; вклад skew при включённой urgency будет меньше.
4. **Detector × ladder — enablement.** Режимы Day19 делают ladder Day18 жизнеспособным в плохие периоды; вклад ladder при включённых режимах будет больше.
5. **Flow × placement — interference.** Лучший fair value меняет, какой уровень «лучший», и сдвигает входные данные placement; вклад placement при включённом flow изменится.

Проверяем это таблицей взаимодействий в §4, а не на глаз.

In [2]:
CFG=dict(flow=True,placement=True,adaptive_width=True,ladder=True,toxicity=True,modes=True,
         hedge_urgency=True,inv_skew=True,alpha=True,hard_limits=True)
def off(**kw):
    c=dict(CFG); c.update(kw); return c

def run(cfg, regime='base', fee_m=0.05, fee_t=0.10, lat=0):
    np.random.seed(SEED)
    buckets=REG[regime]; inv=0.0; fills=[]; quotes=[]; invpath=[]; eq={}
    for i in range(max(W,3000),n-HOLD-lat-1,STEP):
        if i//3000 not in buckets: continue
        if not np.isfinite(h[i]) or not np.isfinite(mu[i]) or not np.isfinite(sig[i]) or not np.isfinite(z[i-1]) or not np.isfinite(volz_t[i]): continue
        bucket=i//3000; volz=volz_t[i]
        mode='normal'
        if cfg['modes']:
            if volz>2.5: mode='paused'
            elif volz>1.5: mode='one-sided'
            elif volz>0.5: mode='wide'
        if mode=='paused': invpath.append(inv); continue
        fair=h[i]*bticks[i]*(BTICK/ETICK)+mu[i]/ETICK
        half_avail=max(0.5,(ea[i-1]-eb[i-1])/2/ETICK)
        if cfg['adaptive_width']:
            mult={'normal':1.0,'wide':1.5,'one-sided':2.0,'recovery':1.2}.get(mode,1.0)
            half=float(np.clip(half_avail*(1+0.8*max(volz,0))*mult,half_avail,5.0))
        else: half=half_avail
        skew=0.0
        if cfg['inv_skew']: skew+=float(np.clip(inv*1e-4*(sig[i]/ETICK)**2*10,-0.6*half,0.6*half))
        if cfg['alpha']:
            stt=0 if abs(z[i-1])<=1.5 else (1 if abs(z[i-1])<=2.5 else 2)
            ma=1.0 if stt==0 else (0.5 if stt==1 else 0.0)
            skew+=float(-0.4*half*np.tanh(z[i-1])*ma)
        if cfg['flow']: skew+=float(np.clip(0.05*ofi[i-1],-0.4*half,0.4*half))
        skew=float(np.clip(skew,-0.9*half,0.9*half))
        k=3 if (cfg['ladder'] and mode=='normal') else (2 if cfg['ladder'] else 1)
        step=2.0; sides=[1,-1]
        if cfg['modes'] and mode=='one-sided': sides=[1] if inv<0 else [-1]
        if cfg['hard_limits'] and abs(inv)>0.5: sides=[1] if inv<0 else [-1]
        j0=np.searchsorted(tt,pt[i],'left'); j1=np.searchsorted(tt,pt[i+STEP],'left')
        for side in sides:
            size_scale=1.0
            if cfg['hard_limits'] and ((inv>0 and side==1) or (inv<0 and side==-1)):
                size_scale=1.0/(1.0+8.0*abs(inv))
            for lev in range(k):
                if cfg['hard_limits'] and ((inv>0.5 and side==1) or (inv<-0.5 and side==-1)): continue
                step_back=cfg['placement'] and (lev>0 or (side==1 and ebs[i-1]>qmed_b[i-1]) or (side==-1 and eas[i-1]>qmed_a[i-1]))
                off_=half+lev*step+(1.0 if step_back else 0.0)
                price=fair-side*off_-side*skew if side==1 else fair+off_-skew
                price=int(np.floor(price+0.5))
                if side==1: price=min(price,int(ea[i-1]/ETICK)-1)
                else: price=max(price,int(eb[i-1]/ETICK)+1)
                tox=abs(z[i-1])+max(volz,0)+0.5*abs(ofi[i-1])/max(ofi_std[i-1],1e-9)
                if cfg['toxicity'] and tox>half+0.5: continue
                touch=price==(int(eb[i-1]/ETICK) if side==1 else int(ea[i-1]/ETICK))
                quotes.append((i,side,lev,price,half,mode,tox,touch))
                if j1<=j0: continue
                ss=sd_e[j0:j1]; pp=px_e[j0:j1]; aa=am_e[j0:j1]
                mask=(ss==(-1 if side==1 else 1)) & ((pp<=price) if side==1 else (pp>=price))
                if not mask.any(): continue
                jj=np.where(mask)[0]
                if touch:
                    Q=ebs[i-1] if side==1 else eas[i-1]
                    cum=np.cumsum(aa[jj]); prob=min(1.0,cum[-1]/max(Q,1e-9))
                    j=jj[np.argmax(cum>=Q)] if cum[-1]>=Q else jj[0]
                else: prob=1.0; j=jj[0]
                kk=int(idx_e[j0+j])
                if kk+HOLD+lat>=n: continue
                T=kk+HOLD+lat; H=min(kk+lat,n-1)
                saved=max(0.0,fee_t-fee_m)*(h[kk]*bt[kk])/10000/ETICK
                risk=0.25*sig[kk]/ETICK
                if cfg['hedge_urgency'] and mode in ('normal','recovery') and saved>risk:
                    H=min(kk+lat+50,n-1)
                ff=h[kk]*bticks[kk]*(BTICK/ETICK)+mu[kk-1]/ETICK
                cap=side*(ff-price); passive=cfg['hedge_urgency'] and mode in ('normal','recovery') and H>kk+lat
                if passive:
                    hp=bticks[kk]; a0=np.searchsorted(ttb,pt[kk],'left'); a1=np.searchsorted(ttb,pt[H],'left'); filled_h=False
                    if a1>a0:
                        ssb=sd_b[a0:a1]; ppb=px_b[a0:a1]; aab=am_b[a0:a1]
                        maskb=(ssb==(1 if side==1 else -1)) & ((ppb>=hp) if side==1 else (ppb<=hp))
                        if maskb.any(): filled_h=True; hex_=hp; hfee=h[kk]*fee_m*bt[kk]/10000/ETICK
                    if not filled_h:
                        hex_=bb[H]/BTICK if side==1 else ba[H]/BTICK; hfee=h[kk]*fee_t*bt[H]/10000/ETICK
                else:
                    hex_=bb[H]/BTICK if side==1 else ba[H]/BTICK; hfee=h[kk]*fee_t*bt[H]/10000/ETICK
                pnl=side*((et[T]/ETICK-price)+h[kk]*(hex_-bticks[T])*(BTICK/ETICK))-fee_m*et[kk]/10000/ETICK-hfee
                spnl=side*((et[T]-h[kk]*bt[T])-(et[H]-h[kk]*bt[H]))/ETICK
                qmark=side*(et[T]/ETICK-price)
                realized=side*(et[kk]/ETICK-price)
                wgt=prob*size_scale
                fills.append((wgt,side,lev,mode,cap,pnl,qmark,spnl,tox,inv,size_scale,half,realized))
                inv+=side*wgt; eq[bucket]=eq.get(bucket,0.0)+wgt*pnl
        invpath.append(inv)
    F=pd.DataFrame(fills,columns=['p','side','level','mode','capture','pnl','qmark','smark','tox','inv_before','size','half','realized'])
    Q=pd.DataFrame(quotes,columns=['i','side','level','price','half','mode','tox','touch'])
    return F,Q,np.array(invpath),eq

def metrics(F,Q,path,eq,buckets,hours):
    if len(F)==0: return dict(fills=0,net_h=0,sharpe=0)
    w=F.p; Qf=w.sum(); net=float(np.sum(w*F.pnl))
    pnl=np.array([eq.get(b,0.0) for b in sorted(buckets)])
    sh=float(pnl.mean()/max(pnl.std(),1e-9)*np.sqrt(365*24*12))
    msg=max(1,len(Q)); peak=int(Q.groupby(Q.i//10).size().max()) if len(Q) else 0
    return dict(fills=len(F),eff=Qf,net_h=net/hours,fills_day=Qf*24/hours,
                capture=float(np.sum(w*F.capture)/Qf),qmark=float(np.sum(w*F.qmark)/Qf),
                smark=float(np.sum(w*F.smark)/Qf),msg_fill=msg/max(len(F),1),
                inv_p95=float(np.percentile(np.abs(path),95)) if len(path) else 0,
                inv_max=float(np.max(np.abs(path))) if len(path) else 0,
                time_above=float(np.mean(np.abs(path)>2)) if len(path) else 0,
                peak_msg_s=peak,headroom=20/max(peak,1),sharpe=sh)

## 3. Абляции: full, девять leave-one-out и Day16-база

Все конфигурации гоняются на одном replay, в base и stressed, при 0.05/0.10 bp. Delta — вклад слоя: full минус конфигурация без него. База — Day16-конфигурация с теми же hard limits и size control, но без flow, placement, width, ladder, toxicity, modes, urgency, skew и alpha.

In [3]:
FM,FT=0.05,0.10
base_cfg=off(flow=False,placement=False,adaptive_width=False,ladder=False,toxicity=False,
             modes=False,hedge_urgency=False,inv_skew=False,alpha=False)
LAYERS=['flow','placement','adaptive_width','ladder','toxicity','modes','hedge_urgency','inv_skew','alpha']
rows=[]; RUNS={}
for reg in ['base','stressed']:
    hours=len(REG[reg])*5/60
    for name,cfg in [('full',CFG),('baseline',base_cfg)]+[('-'+k,off(**{k:False})) for k in LAYERS]:
        F,Q,path,eq=run(cfg,reg,fee_m=FM,fee_t=FT); RUNS[(reg,name)]=(F,Q,path,eq)
        m=metrics(F,Q,path,eq,REG[reg],hours); rows.append({'regime':reg,'config':name,**m})
AB=pd.DataFrame(rows)
piv=AB.pivot(index='config',columns='regime',values='net_h').round(1)
display(piv)
print('Delta слоя (full - без слоя), тиков/час:')
for k in LAYERS:
    print('  %-16s base %+8.1f   stressed %+8.1f'%(k,piv.loc['full','base']-piv.loc['-'+k,'base'],piv.loc['full','stressed']-piv.loc['-'+k,'stressed']))
display(AB[['regime','config','fills','eff','net_h','capture','smark','inv_p95','msg_fill','sharpe']].round(2))

regime,base,stressed
config,,
-adaptive_width,169.8,293.8
-alpha,116.4,946.4
-flow,176.0,1052.6
-hedge_urgency,180.5,1047.8
-inv_skew,155.2,1117.8
-ladder,137.3,424.2
-modes,239.7,316.1
-placement,111.2,996.3
-toxicity,528.1,1183.4


Delta слоя (full - без слоя), тиков/час:
  flow             base     +4.5   stressed     -4.8
  placement        base    +69.3   stressed    +51.5
  adaptive_width   base    +10.7   stressed   +754.0
  ladder           base    +43.2   stressed   +623.6
  toxicity         base   -347.6   stressed   -135.6
  modes            base    -59.2   stressed   +731.7
  hedge_urgency    base     +0.0   stressed     +0.0
  inv_skew         base    +25.3   stressed    -70.0
  alpha            base    +64.1   stressed   +101.4


,regime,config,fills,eff,net_h,capture,smark,inv_p95,msg_fill,sharpe
0,base,full,239,154.45,180.47,3.99,0.18,1.38,1.90,86.03
1,base,baseline,342,173.17,-225.44,3.43,-0.67,0.92,1.81,-117.10
2,base,-flow,239,154.45,176.03,3.97,0.17,1.38,1.90,83.51
3,base,-placement,241,140.98,111.20,3.97,0.40,1.40,1.91,93.99
4,base,-adaptive_width,180,114.20,169.77,4.85,0.47,1.41,1.98,92.75
5,base,-ladder,149,80.96,137.27,3.11,0.97,0.73,1.59,187.37
6,base,-toxicity,528,292.82,528.14,4.50,0.45,1.41,2.26,122.87
7,base,-modes,204,133.69,239.68,4.16,0.96,1.46,1.95,109.09
8,base,-hedge_urgency,239,154.45,180.47,3.99,0.18,1.38,1.90,86.03
9,base,-inv_skew,239,160.20,155.17,3.86,0.06,1.44,1.87,97.22


## 4. Baseline, interaction gap и предсказания

Сравниваем full с Day16-базой по метрикам, затем считаем сумму leave-one-out дельт и фактический разрыв. Если сумма дельт не равна разрыву, остаток — взаимодействия. Для пяти предсказанных пар считаем 2×2 interaction: $I=f(1,1)-f(1,0)-f(0,1)+f(0,0)$; положительный $I$ — enablement, отрицательный — substitution/interference.

В base flow и adaptive width дают небольшой плюс, toxicity и modes — минус, но full всё равно остаётся лучше базы: interaction gap +596 — слои дополняют друг друга, и выключение одного не изолирует его вклад. В stressed сумма дельт +2052, фактический разрыв +602, gap −1450: width (+754), modes (+732) и ladder (+624) сильно перекрываются, а toxicity (−136) частично их перекрывает. В crisis full −705 против базы −605, и все тестируемые пары interactions отрицательны (urgency×skew = 0, не тестируется): double application width×modes (−1020), toxicity×ladder (−857), modes×ladder (−385). Это и есть «интеграция — не сложение»: знак interaction зависит от режима.

**Счёт предсказаний.** 1) toxicity × ladder — substitution подтверждён: −565 / −258 / −857. 2) width × modes — double application подтверждён в crisis (−1020), но в stressed знак положительный (+732); вывод режимный. 3) urgency × skew — не тестируется на выбранном тире: urgency не включается, потому что сэкономленная комиссия меньше residual-риска; interaction 0. 4) detector × ladder — эффект включения **не подтверждён**: −86 / +437 / −385, устойчивого положительного эффекта нет. 5) flow × placement — interference подтверждён сменой знака: −58 / +19 / −22. Итог: два предсказания подтверждены, одно режимное, одно не тестируется, одно не подтвердилось.

In [4]:
print('Day16 baseline vs full:')
for reg in ['base','stressed']:
    b=AB[(AB.regime==reg)&(AB.config=='baseline')].iloc[0]; f=AB[(AB.regime==reg)&(AB.config=='full')].iloc[0]
    print('%s: net/h %+.1f vs %+.1f; fills/day %.0f vs %.0f; capture %.2f vs %.2f; smark %.2f vs %.2f; inv p95 %.2f vs %.2f; msg/fill %.1f vs %.1f'%(
        reg,b.net_h,f.net_h,b.fills_day,f.fills_day,b.capture,f.capture,b.smark,f.smark,b.inv_p95,f.inv_p95,b.msg_fill,f.msg_fill))
print('\nInteraction gap (full - baseline):')
for reg in ['base','stressed']:
    actual=AB[(AB.regime==reg)&(AB.config=='full')].net_h.iloc[0]-AB[(AB.regime==reg)&(AB.config=='baseline')].net_h.iloc[0]
    s=sum(AB[(AB.regime==reg)&(AB.config=='full')].net_h.iloc[0]-AB[(AB.regime==reg)&(AB.config=='-'+k)].net_h.iloc[0] for k in LAYERS)
    print('  %s: actual %+.1f, sum deltas %+.1f, gap %+.1f'%(reg,actual,s,actual-s))

PAIRS=[('toxicity','ladder'),('adaptive_width','modes'),('hedge_urgency','inv_skew'),('modes','ladder'),('flow','placement')]
pair_rows=[]
for reg in ['base','stressed']:
    f11=AB[(AB.regime==reg)&(AB.config=='full')].net_h.iloc[0]
    for a,b_ in PAIRS:
        f10=AB[(AB.regime==reg)&(AB.config=='-'+a)].net_h.iloc[0]
        f01=AB[(AB.regime==reg)&(AB.config=='-'+b_)].net_h.iloc[0]
        key=(reg,'-%s-%s'%(a,b_))
        if key not in RUNS:
            F,Q,path,eq=run(off(**{a:False,b_:False}),reg,fee_m=FM,fee_t=FT); RUNS[key]=(F,Q,path,eq)
        f00=metrics(*RUNS[key],REG[reg],len(REG[reg])*5/60)['net_h']
        pair_rows.append({'regime':reg,'pair':'%s x %s'%(a,b_),'interaction':f11-f10-f01+f00})
PI=pd.DataFrame(pair_rows).pivot(index='pair',columns='regime',values='interaction').round(1)
display(PI)
print('Сумма парных interactions — часть общего gap; остаток — взаимодействия высшего порядка.')
# crisis sanity + pair interactions for the predictions that only fire in crisis
Fc,Qc,pc,ec=run(CFG,'crisis',fee_m=FM,fee_t=FT); Bc,Qbc,pbc,ebc=run(base_cfg,'crisis',fee_m=FM,fee_t=FT)
print('\nCrisis: full %+.1f vs baseline %+.1f тиков/ч'%(
    metrics(Fc,Qc,pc,ec,REG['crisis'],len(REG['crisis'])*5/60)['net_h'],
    metrics(Bc,Qbc,pbc,ebc,REG['crisis'],len(REG['crisis'])*5/60)['net_h']))
crisis_rows=[]; f11=metrics(Fc,Qc,pc,ec,REG['crisis'],len(REG['crisis'])*5/60)['net_h']
for a,b_ in PAIRS:
    f10=metrics(*run(off(**{a:False}),'crisis',fee_m=FM,fee_t=FT),REG['crisis'],len(REG['crisis'])*5/60)['net_h']
    f01=metrics(*run(off(**{b_:False}),'crisis',fee_m=FM,fee_t=FT),REG['crisis'],len(REG['crisis'])*5/60)['net_h']
    f00=metrics(*run(off(**{a:False,b_:False}),'crisis',fee_m=FM,fee_t=FT),REG['crisis'],len(REG['crisis'])*5/60)['net_h']
    crisis_rows.append({'pair':'%s x %s'%(a,b_),'interaction':f11-f10-f01+f00})
display(pd.DataFrame(crisis_rows).set_index('pair').round(1))

Day16 baseline vs full:
base: net/h -225.4 vs +180.5; fills/day 6234 vs 5560; capture 3.43 vs 3.99; smark -0.67 vs 0.18; inv p95 0.92 vs 1.38; msg/fill 1.8 vs 1.9
stressed: net/h +445.5 vs +1047.8; fills/day 8304 vs 5940; capture 4.86 vs 2.51; smark 1.73 vs 4.89; inv p95 0.88 vs 1.42; msg/fill 1.6 vs 1.6

Interaction gap (full - baseline):
  base: actual +405.9, sum deltas -189.9, gap +595.8
  stressed: actual +602.3, sum deltas +2051.9, gap -1449.6


regime,base,stressed
pair,,
adaptive_width x modes,-15.2,731.7
flow x placement,-58.1,19.4
hedge_urgency x inv_skew,0.0,0.0
modes x ladder,-85.9,437.4
toxicity x ladder,-565.3,-258.3


Сумма парных interactions — часть общего gap; остаток — взаимодействия высшего порядка.

Crisis: full -705.2 vs baseline -605.0 тиков/ч


,interaction
pair,
toxicity x ladder,-857.2
adaptive_width x modes,-1020.4
hedge_urgency x inv_skew,0.0
modes x ladder,-385.4
flow x placement,-22.3


## 5. Шесть отчётов из лога

Считаем по полному прогону на всём двухчасовом окне (24 бакета) при 0.05/0.10 bp. Fill, capture и markout — по логу филлов; inventory tail — time-weighted по пути инвентаря; cancel efficiency — по логу котировок; latency sensitivity — отдельные прогоны с задержкой 0–16 шагов. Всё, кроме latency, пересчитывается из сохранённого лога без перезапуска стратегии.

In [5]:
F,Q,path,eq=run(CFG,'all',fee_m=FM,fee_t=FT)
w=F.p; Qf=w.sum()
print('FILL: fills %d, effective %.1f, quotes %d, fill/quote %.3f, at-touch share %.2f'%(
    len(F),Qf,len(Q),len(F)/max(len(Q),1),Q.touch.mean()))
display(F.groupby(['mode','level']).agg(fills=('p','size'),eff=('p','sum'),capture=('capture','mean')).round(2))
print('attempted vs achieved size: 1.00 vs %.2f (eff/fills)'%(Qf/len(F)))
print('\nCAPTURE: vs fair %.2f ticks, realized vs quoted mid %.2f, intended half %.2f, realized share %.2f'%(
    np.sum(w*F.capture)/Qf,np.sum(w*F.realized)/Qf,np.sum(w*F.half)/Qf,np.sum(w*F.realized)/np.sum(w*F.half)))
display(F.groupby(['mode','level']).apply(lambda d: pd.Series({'capture_fair':np.sum(d.p*d.capture)/d.p.sum(),'realized':np.sum(d.p*d.realized)/d.p.sum(),'half':np.sum(d.p*d.half)/d.p.sum()})).round(2))
print('Capture vs fair может превышать half, когда fair уходит от mid котируемой ноги; экономический захват — realized vs quoted mid и spread markout.')
print('\nMARKOUT: quoted %.2f, spread %.2f, gap %.2f ticks'%(
    np.sum(w*F.qmark)/Qf,np.sum(w*F.smark)/Qf,np.sum(w*(F.qmark-F.smark))/Qf))
F['tox_dec']=pd.qcut(F.tox,5,labels=False,duplicates='drop')
mk=F.groupby('tox_dec').apply(lambda d: pd.Series({'n':len(d),'smark':np.sum(d.p*d.smark)/d.p.sum(),'se':np.std(d.smark)/np.sqrt(len(d))})).round(2)
display(mk)

FILL: fills 578, effective 404.4, quotes 979, fill/quote 0.590, at-touch share 0.39


fills    eff  capture
mode      level                       
normal    0        140  94.55     4.35
          1        121  91.52     5.33
          2         39  17.13    11.89
one-sided 0         74  57.09    11.44
          1         56  44.75    13.03
wide      0         83  54.60     3.97
          1         65  44.73     5.49

attempted vs achieved size: 1.00 vs 0.70 (eff/fills)

CAPTURE: vs fair 4.88 ticks, realized vs quoted mid 2.95, intended half 1.34, realized share 2.20


capture_fair  realized  half
mode      level                              
normal    0              2.04      2.65  0.64
          1              3.06      3.41  0.64
          2              8.23      2.58  0.52
one-sided 0             10.42      3.54  2.80
          1             11.51      5.59  2.74
wide      0              1.87      1.48  1.34
          1              3.33      1.16  1.33

Capture vs fair может превышать half, когда fair уходит от mid котируемой ноги; экономический захват — realized vs quoted mid и spread markout.

MARKOUT: quoted -13.75, spread -0.24, gap -13.51 ticks


,n,smark,se
tox_dec,,,
0,116.0,-1.85,1.85
1,116.0,1.45,1.75
2,116.0,0.51,2.07
3,115.0,2.12,2.20
4,115.0,-3.08,3.73


In [6]:
print('INVENTORY TAIL (time-weighted): p50 %.2f, p95 %.2f, p99 %.2f, max %.2f, time>|2| %.3f'%(
    np.percentile(np.abs(path),50),np.percentile(np.abs(path),95),np.percentile(np.abs(path),99),
    np.max(np.abs(path)),np.mean(np.abs(path)>2)))
print('Funding stamps 00/08/16 UTC: в окне 12:00–14:00 их нет, позиция на штампе не наблюдается.')
plc=len(Q); cnc=max(0,plc-len(F)); peak=int(Q.groupby(Q.i//10).size().max())
print('\nCANCEL EFFICIENCY: placements %d, implied cancels %d, total messages %d, msg/fill %.2f (with cancels), placement peak %d/s, total peak ~%d/s, cap 20/s'%(
    plc,cnc,plc+cnc,(plc+cnc)/max(len(F),1),peak,2*peak))
L=[]
for lat in [0,1,2,4,8,16]:
    Fl,Ql,pl,el=run(CFG,'all',fee_m=FM,fee_t=FT,lat=lat)
    L.append({'latency, x100ms':lat,'net/h':metrics(Fl,Ql,pl,el,REG['all'],2.0)['net_h'],'fills':len(Fl)})
LAT=pd.DataFrame(L); display(LAT.round(1))
print('Zero crossing (model convention: delays mark/hedge, not quote placement; treat as horizon decay): ', end='')
neg=LAT[LAT['net/h']<=0]
print('>16x (headroom)' if len(neg)==0 else '%.0fx'%neg.iloc[0]['latency, x100ms'])

INVENTORY TAIL (time-weighted): p50 0.98, p95 1.42, p99 1.48, max 1.49, time>|2| 0.000
Funding stamps 00/08/16 UTC: в окне 12:00–14:00 их нет, позиция на штампе не наблюдается.

CANCEL EFFICIENCY: placements 979, implied cancels 401, total messages 1380, msg/fill 2.39 (with cancels), placement peak 6/s, total peak ~12/s, cap 20/s


,"latency, x100ms",net/h,fills
0,0,43.3,578
1,1,-681.9,578
2,2,-1621.7,578
3,4,-1865.4,578
4,8,-2129.9,578
5,16,-2463.1,578


Zero crossing (model convention: delays mark/hedge, not quote placement; treat as horizon decay): 1x


## 6. Оптимизация по тарифам

Перебираем maker/taker в bp на полном двухчасовом окне. Для каждой пары считаем net/h и in-sample Sharpe по 24 пятиминутным бакетам. Задача — найти тир, где Sharpe ≥ 1, и понять, существует ли он на бирже. Sharpe здесь in-sample и на коротком окне, поэтому это верхняя оценка, а не прогноз.

In [7]:
FE=[]
for fm in [0,0.02,0.05,0.1,0.2]:
    for ft in [0,0.05,0.1,0.2,0.5,1.0,2.0]:
        Fl,Ql,pl,el=run(CFG,'all',fee_m=fm,fee_t=ft)
        m=metrics(Fl,Ql,pl,el,REG['all'],2.0)
        FE.append({'maker, bp':fm,'taker, bp':ft,'net/h':m['net_h'],'sharpe':m['sharpe']})
FE=pd.DataFrame(FE)
display(FE.pivot(index='maker, bp',columns='taker, bp',values='net/h').round(0))
ok=FE[(FE.sharpe>=1)&(FE['net/h']>0)].sort_values(['maker, bp','taker, bp'])
print('Тиры с Sharpe >= 1:')
display(ok.round(2))
if len(ok):
    cheap=ok.sort_values(['taker, bp','maker, bp']).iloc[-1]
    print('Самый дорогой по taker тир с Sharpe>=1: maker %.2f bp, taker %.2f bp, net/h %.0f, Sharpe %.1f'%(
        cheap['maker, bp'],cheap['taker, bp'],cheap['net/h'],cheap['sharpe']))
print('При retail 0.1/2 bp стратегия убыточна; тир 0.05/0.10 — верхний MM-тир, не retail.')

"taker, bp",0.00,0.05,0.10,0.20,0.50,1.00,2.00
"maker, bp",,,,,,,
0.00,520.0,361.0,204.0,-419.0,-2344.0,-3547.0,-5941.0
0.02,456.0,297.0,139.0,-346.0,-2414.0,-3626.0,-6020.0
0.05,361.0,202.0,43.0,-353.0,-2536.0,-3745.0,-6139.0
0.10,202.0,43.0,-116.0,-432.0,-2677.0,-3943.0,-6337.0
0.20,-116.0,-274.0,-433.0,-751.0,-2736.0,-4339.0,-6733.0


Тиры с Sharpe >= 1:


,"maker, bp","taker, bp",net/h,sharpe
0,0.00,0.00,519.80,107.34
1,0.00,0.05,360.96,75.35
2,0.00,0.10,203.83,42.82
7,0.02,0.00,456.26,94.67
8,0.02,0.05,297.42,62.28
9,0.02,0.10,138.59,29.17
14,0.05,0.00,360.96,75.34
15,0.05,0.05,202.12,42.47
16,0.05,0.10,43.28,9.12
21,0.10,0.00,202.11,42.47


Самый дорогой по taker тир с Sharpe>=1: maker 0.05 bp, taker 0.10 bp, net/h 43, Sharpe 9.1
При retail 0.1/2 bp стратегия убыточна; тир 0.05/0.10 — верхний MM-тир, не retail.


## 7. Acceptance, Protocol и журнал оптимизма

| Критерий | Результат |
|---|---|
| Positive net in stressed | да: +1048 тиков/ч при 0.05/0.10 |
| Beats Day16 in both modes | да: base +180 против −225; stressed +1048 против +446 |
| Layers switchable independently | да: девять leave-one-out прогонов |
| Interactions identified and explained | да: gap +596 / −1450; парные interactions по трём режимам |
| Six reports from the log | да: один лог полного прогона, latency — отдельные прогоны |
| Inventory tail no wider than baseline | **нет**: p95 1.42 против 0.88 ETH в stressed; часть прироста — leverage |
| Effective independent trades ≥ 100 in stressed | да: 225 филлов, ~160 effective по prob-весам |
| Rate-limit headroom | да: placement peak 6/s; с подразумеваемыми отменами ~12/s при cap 20 |
| Protocol complete | полностью: десять полей ниже |
| Crisis sanity | **нет**: full −705 против базы −605; режимный overlay в crisis не подтверждён |
| Latency headroom | model convention: задерживается mark/hedge, не выставление котировки; zero crossing 1× — оценка сверху, не полный latency-backtest |

**Protocol (десять полей).**
1. Механизм: single-venue ETH/BTC, котируем ETH, хеджируем BTC ratio 1:1 по нотионалу.
2. Инструменты: Bybit ETHUSDT / BTCUSDT perpetual.
3. Данные: 2026-07-01 12:00–14:00 UTC, 100-мс top-of-book и трейды.
4. Тариф: 0.05 bp maker / 0.10 bp taker (выбранный верхний тир); retail 0.1/2 bp — убыток.
5. Задержка: 1× = 100 мс; чувствительность 0–16×; модель задерживает выход, не постановку.
6. Бюджет сообщений: 20 msg/s; placement peak 6/s, с отменами ~12/s.
7. Инвентарь: soft target 0.5 ETH, size scaling $1/(1+8|inv|)$; наблюдаемый p95 1.42 — это не жёсткий кап.
8. Режим отказа: trailing volz > 2.5 или paused-overlay; наблюдаемо по 5-мин vol z и |z_spread|.
9. Критерий отклонения в числах: Sharpe < 1, net stressed ≤ 0, tail > baseline, effective trades < 100.
10. Мониторинг: шесть отчётов ежедневно из лога, latency-кривая еженедельно.

**Журнал оптимизма.**
- Fill-модель оптимистична: очередь моделируется долей съеденного объёма, без позиции в очереди и скрытой ликвидности.
- Top-of-book, без полного стакана и market impact; размер 1 ETH.
- Ratio-хедж 1:1 по нотионалу, без funding (в окне нет штампов) и без borrow.
- Flow-терм — mid-momentum proxy, не depth-OFI.
- Sharpe in-sample на 24 бакетах — верхняя оценка; в таблицах это не прогноз.
- Тариф 0.05/0.10 — предположение о верхнем MM-тире; на retail стратегия отрицательна.
- Crisis-стратификация и trailing-vol сигнал частично circular: бакет выбран по realized vol, а стратегия видит trailing vol; в crisis это может запаздывать.
- Направление смещения: net скорее завышен, хвост инвентаря скорее занижен. План: queue reconstruction, funding, multi-venue replay.

## 8. Вывод

Интеграция собрана и измерима: десять переключателей, девять абляций, шесть отчётов, baseline и interaction gap. На верхнем тире стратегия положительна в base и stressed и превосходит Day16-базу, но **не проходит два жёстких пункта**: хвост инвентаря p95 1.42 против 0.88 ETH, и в crisis она хуже базы (−705 против −605). Латентностная кривая в этой модели — задержка выхода, а не постановки котировки; zero crossing 1× — оценка сверху, но запаса всё равно не видно. Значит, «больше net» здесь частично является leverage и regime-specific эффектом, а не устойчивым улучшением.

Главный урок: дельты не складываются и меняют знак по режимам. В base сумма leave-one-out дельт −190, фактический разрыв +406: слои дополняют друг друга, и выключение одного не изолирует вклад. В stressed сумма +2052, фактический +602: width, modes и ladder сильно перекрываются, toxicity их частично перекрывает. В crisis все тестируемые пары отрицательны (urgency×skew = 0, не тестируется) — double application и interference сильнее любого отдельного улучшения. Это ответ на вопрос «какая часть несёт результат»: в base — комбинация, в stressed — width/modes/ladder, в crisis — ни один слой из набора не вытягивает; inventory skew в base даёт +25, в stressed −70, то есть в несущие слои stressed не входит.

Тарифы: Sharpe ≥ 1 проходит при taker ≤ 0.1 bp и maker ≤ 0.05 bp либо при taker ≤ 0.05 bp и maker ≤ 0.10 bp; при 0.1/0.1 bp net уже −116 тиков/ч, при retail 0.1/2 bp — глубоко отрицателен. Для production нужен либо верхний MM-тир, либо более широкая котируемая нога, либо отказ от crisis-режима. Следующий шаг — пересчитать crisis-ветку с менее агрессивным gate и проверить, сохраняется ли положительный stressed-net после ужесточения inventory tail.